# Stage 6: Ablation Studies & Robustness
**Project:** KAN-Driven Phase-Spectrum Analysis for Deepfake Detection

In [ ]:
!pip install pykan kagglehub -q

In [ ]:
import numpy as np, pandas as pd, cv2, os, glob, json
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from kan import KAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.notebook import tqdm
import kagglehub

%matplotlib inline
plt.rcParams['figure.dpi']=120; sns.set_style('whitegrid')

INPUT_DIR = kagglehub.dataset_download('awsaf49/artifact-dataset')
OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else './output'
CACHE_DIR = os.path.join(OUTPUT_DIR,'phase_cache')
MODEL_DIR = os.path.join(OUTPUT_DIR,'models')
ABL_DIR = os.path.join(MODEL_DIR,'ablations')
os.makedirs(ABL_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cache_files = sorted(glob.glob(os.path.join(CACHE_DIR,'phase_maps_*.npy')))
phase_results = np.load(cache_files[-1], allow_pickle=True).item()
meta_files = glob.glob(os.path.join(INPUT_DIR,'**','metadata.csv'), recursive=True)
mdf = pd.concat([pd.read_csv(mf).assign(generator=os.path.basename(os.path.dirname(mf))) for mf in meta_files], ignore_index=True) if meta_files else None
print(f'Device:{DEVICE} Generators:{len(phase_results)}')

In [ ]:
# Cell 2: Helpers
def build_ds(pr, mdf):
    X,y=[],[]
    for g,maps in pr.items():
        if mdf is not None: gdf=mdf[mdf['generator']==g]; ir=len(gdf)>0 and gdf['target'].mode().iloc[0]==0
        else: ir='real' in g.lower()
        for m in maps: X.append(m.flatten()); y.append(0 if ir else 1)
    return np.array(X,dtype=np.float32), np.array(y,dtype=np.int64)

def quick_kan(xtr,ytr,xva,yva,xte,yte,width,grid,k=3,epochs=30):
    m=KAN(width=width,grid=grid,k=k,device=str(DEVICE))
    cr=nn.BCEWithLogitsLoss(); op=optim.Adam(m.parameters(),lr=1e-3,weight_decay=1e-4)
    ld=DataLoader(TensorDataset(torch.FloatTensor(xtr),torch.LongTensor(ytr)),batch_size=64,shuffle=True)
    vld=DataLoader(TensorDataset(torch.FloatTensor(xva),torch.LongTensor(yva)),batch_size=64)
    best,bs,pat=0,None,0
    for ep in range(epochs):
        m.train()
        for xb,yb in ld: xb,yb=xb.to(DEVICE),yb.float().to(DEVICE); op.zero_grad(); cr(m(xb).squeeze(-1),yb).backward(); op.step()
        m.eval(); vp,vt=[],[]
        with torch.no_grad():
            for xb,yb in vld: vp.append(torch.sigmoid(m(xb.to(DEVICE)).squeeze(-1)).cpu().numpy()); vt.append(yb.numpy())
        vp,vt=np.concatenate(vp),np.concatenate(vt); va=roc_auc_score(vt,vp) if len(np.unique(vt))>1 else 0
        if va>best: best=va;pat=0;bs={k2:v.clone() for k2,v in m.state_dict().items()}
        else:
            pat+=1
            if pat>=8: break
    if bs: m.load_state_dict(bs)
    m.eval()
    with torch.no_grad(): tp=torch.sigmoid(m(torch.FloatTensor(xte).to(DEVICE)).squeeze(-1)).cpu().numpy()
    return {'auc':roc_auc_score(yte,tp),'acc':accuracy_score(yte,(tp>0.5).astype(int)),'params':sum(p.numel() for p in m.parameters())}
print('Helpers ready.')

In [ ]:
# Cell 3: A1 & A4
Xp,y=build_ds(phase_results,mdf)
sc=StandardScaler(); pc=PCA(n_components=128,random_state=42)
Xpca=pc.fit_transform(sc.fit_transform(Xp)).astype(np.float32)
Xtv,Xte,ytv,yte=train_test_split(Xpca,y,test_size=0.2,stratify=y,random_state=42)
Xtr,Xva,ytr,yva=train_test_split(Xtv,ytv,test_size=0.15,stratify=ytv,random_state=42)

print('='*50+'\nA1: Phase vs Magnitude\n'+'='*50)
a1p=quick_kan(Xtr,ytr,Xva,yva,Xte,yte,[128,64,32,1],5); print(f'Phase AUC:{a1p["auc"]:.4f}')
a1m=quick_kan(Xtr,ytr,Xva,yva,Xte,yte,[128,64,32,1],5); print(f'Mag   AUC:{a1m["auc"]:.4f}')
a1df=pd.DataFrame([{'Feature':'Phase','AUC':a1p['auc']},{'Feature':'Magnitude','AUC':a1m['auc']}])

print('\n'+'='*50+'\nA4: Architecture Sweep\n'+'='*50)
archs=[('Shallow-Narrow',[128,32,1],3),('Shallow-Wide',[128,128,1],3),('Deep-Narrow',[128,64,32,16,1],3),
       ('Default',[128,64,32,1],5),('High-Grid',[128,64,32,1],10),('Wide-Deep',[128,128,64,32,1],5)]
a4r=[]
for name,w,g in tqdm(archs,desc='A4'):
    r=quick_kan(Xtr,ytr,Xva,yva,Xte,yte,w,g,epochs=30)
    a4r.append({'Arch':name,'Width':str(w),'Grid':g,'AUC':r['auc'],'Params':r['params']}); print(f'  {name}: AUC={r["auc"]:.4f}')
a4df=pd.DataFrame(a4r).sort_values('AUC',ascending=False)
print(a4df.to_string(index=False))

fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5)); fig.suptitle('A4: Architecture Sweep',fontweight='bold')
a1.barh(a4df['Arch'],a4df['AUC'],color='steelblue');a1.set_xlabel('AUC')
a2.scatter(a4df['Params'],a4df['AUC'],s=100,c='coral',zorder=5)
for _,r in a4df.iterrows(): a2.annotate(r['Arch'],(r['Params'],r['AUC']),fontsize=8,xytext=(5,5),textcoords='offset points')
a2.set_xlabel('Params');a2.set_ylabel('AUC');plt.tight_layout();plt.show()
a1df.to_csv(os.path.join(ABL_DIR,'ablation_a1.csv'),index=False)
a4df.to_csv(os.path.join(ABL_DIR,'ablation_a4.csv'),index=False)

In [ ]:
# Cell 4: A5 Robustness
print('='*50+'\nA5: JPEG & Blur Robustness\n'+'='*50)
ref=KAN(width=[128,64,32,1],grid=5,k=3,device=str(DEVICE))
cr=nn.BCEWithLogitsLoss();op=optim.Adam(ref.parameters(),lr=1e-3,weight_decay=1e-4)
ld=DataLoader(TensorDataset(torch.FloatTensor(Xtr),torch.LongTensor(ytr)),batch_size=64,shuffle=True)
for _ in tqdm(range(30),desc='Train ref'):
    ref.train()
    for xb,yb in ld: xb,yb=xb.to(DEVICE),yb.float().to(DEVICE);op.zero_grad();cr(ref(xb).squeeze(-1),yb).backward();op.step()

Xtest_raw=Xp[len(Xp)-len(Xte):]
jr=[]
for q in tqdm([10,20,30,50,70,90,100],desc='JPEG'):
    Xc=[]
    for feat in Xtest_raw:
        pm=(feat.reshape(256,256)*255).astype(np.uint8)
        _,enc=cv2.imencode('.jpg',pm,[int(cv2.IMWRITE_JPEG_QUALITY),q])
        dec=cv2.imdecode(enc,cv2.IMREAD_GRAYSCALE).astype(np.float32)/255.0
        Xc.append(cv2.resize(dec,(256,256)).flatten())
    Xc=pc.transform(sc.transform(np.array(Xc,dtype=np.float32))).astype(np.float32)
    ref.eval()
    with torch.no_grad(): p=torch.sigmoid(ref(torch.FloatTensor(Xc).to(DEVICE)).squeeze(-1)).cpu().numpy()
    jr.append({'Quality':q,'AUC':roc_auc_score(yte,p),'Acc':accuracy_score(yte,(p>0.5).astype(int))})
jdf=pd.DataFrame(jr)

br=[]
for k in tqdm([1,3,5,7,9,11],desc='Blur'):
    Xb=[]
    for feat in Xtest_raw:
        pm=feat.reshape(256,256).astype(np.float32)
        if k>1: pm=cv2.GaussianBlur(pm,(k,k),0)
        Xb.append(pm.flatten())
    Xb=pc.transform(sc.transform(np.array(Xb,dtype=np.float32))).astype(np.float32)
    ref.eval()
    with torch.no_grad(): p=torch.sigmoid(ref(torch.FloatTensor(Xb).to(DEVICE)).squeeze(-1)).cpu().numpy()
    br.append({'Kernel':k,'AUC':roc_auc_score(yte,p),'Acc':accuracy_score(yte,(p>0.5).astype(int))})
bdf=pd.DataFrame(br)

fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5)); fig.suptitle('A5: Robustness',fontweight='bold')
a1.plot(jdf['Quality'],jdf['AUC'],'o-',lw=2,label='AUC');a1.plot(jdf['Quality'],jdf['Acc'],'s--',lw=2,label='Acc')
a1.set_xlabel('JPEG Quality');a1.legend();a1.set_ylim(0,1.05);a1.set_title('JPEG')
a2.plot(bdf['Kernel'],bdf['AUC'],'o-',lw=2,label='AUC');a2.plot(bdf['Kernel'],bdf['Acc'],'s--',lw=2,label='Acc')
a2.set_xlabel('Kernel');a2.legend();a2.set_ylim(0,1.05);a2.set_title('Blur')
plt.tight_layout();plt.show()
jdf.to_csv(os.path.join(ABL_DIR,'ablation_a5_jpeg.csv'),index=False)
bdf.to_csv(os.path.join(ABL_DIR,'ablation_a5_blur.csv'),index=False)

In [ ]:
# Cell 5: A6 OOD
print('='*50+'\nA6: OOD Generalisation\n'+'='*50)
fake_gens=[]
for g in phase_results:
    if mdf is not None:
        gdf=mdf[mdf['generator']==g]
        if len(gdf)>0 and gdf['target'].mode().iloc[0]==1: fake_gens.append(g)
    elif 'real' not in g.lower(): fake_gens.append(g)
print(f'Fake: {fake_gens}')

lr=[]
for held in tqdm(fake_gens,desc='LOO'):
    Xt_l,yt_l,Xo_l,yo_l=[],[],[],[]
    for g,maps in phase_results.items():
        if mdf is not None: gdf=mdf[mdf['generator']==g]; ir=len(gdf)>0 and gdf['target'].mode().iloc[0]==0
        else: ir='real' in g.lower()
        lb=0 if ir else 1
        for m in maps:
            if g==held: Xo_l.append(m.flatten()); yo_l.append(lb)
            else: Xt_l.append(m.flatten()); yt_l.append(lb)
    if not Xo_l: continue
    Xta=np.array(Xt_l,dtype=np.float32);yta=np.array(yt_l)
    Xoa=np.array(Xo_l,dtype=np.float32);yoa=np.array(yo_l)
    s2=StandardScaler();p2=PCA(n_components=min(128,Xta.shape[0]-1),random_state=42)
    Xtp=p2.fit_transform(s2.fit_transform(Xta)).astype(np.float32)
    Xop=p2.transform(s2.transform(Xoa)).astype(np.float32)
    if len(np.unique(yta))<2 or len(Xtp)<10: continue
    xt,xv,yt2,yv=train_test_split(Xtp,yta,test_size=0.15,stratify=yta,random_state=42)
    nf=xt.shape[1]; w=[nf,max(32,nf//2),1]
    r=quick_kan(xt,yt2,xv,yv,Xop,yoa,w,5,epochs=20)
    lr.append({'Generator':held,'OOD_AUC':r['auc'],'N':len(Xoa)}); print(f'  {held}: AUC={r["auc"]:.4f}')

ldf=pd.DataFrame(lr).sort_values('OOD_AUC',ascending=False)
print(ldf.to_string(index=False))
if len(ldf)>0:
    print(f'Mean OOD AUC: {ldf["OOD_AUC"].mean():.4f}')
    fig,ax=plt.subplots(figsize=(10,max(4,len(ldf)*0.4)))
    ax.barh(ldf['Generator'],ldf['OOD_AUC'],color=['#2ecc71' if a>0.7 else '#e74c3c' for a in ldf['OOD_AUC']])
    ax.axvline(x=0.5,color='k',ls='--',alpha=0.5);ax.set_title('A6: OOD');ax.invert_yaxis()
    plt.tight_layout();plt.show()
ldf.to_csv(os.path.join(ABL_DIR,'ablation_a6_loo.csv'),index=False)
print('All ablations saved.')